# HPC Lab — Experiment 10
## CUDA Matrix Multiplication, Reduction, Scan & Histogram

**Name:** Aditi Mittal | **PRN:** 23610067 | **Batch:** B1

---
> **Before running:** Go to `Runtime → Change runtime type → Hardware accelerator → T4 GPU`

In [ ]:
!nvidia-smi

---
## Task 1a — CPU Sequential Matrix Multiplication
Baseline: `ikj` loop order for sizes 256, 512, 1024.

In [ ]:
%%writefile task1_matmul_cpu.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

void matmul_cpu(float *A, float *B, float *C, int n) {
    for (int i=0; i<n; i++)
        for (int j=0; j<n; j++) {
            float sum = 0.0f;
            for (int k=0; k<n; k++) sum += A[i*n+k] * B[k*n+j];
            C[i*n+j] = sum;
        }
}

int main() {
    int sizes[] = {256, 512, 1024};
    for (int s=0; s<3; s++) {
        int n = sizes[s];
        size_t bytes = n*n*sizeof(float);
        float *A=(float*)malloc(bytes), *B=(float*)malloc(bytes), *C=(float*)malloc(bytes);
        for (int i=0; i<n*n; i++) { A[i]=rand()%10; B[i]=rand()%10; }
        struct timespec ts, te;
        clock_gettime(CLOCK_MONOTONIC, &ts);
        matmul_cpu(A, B, C, n);
        clock_gettime(CLOCK_MONOTONIC, &te);
        double ms = (te.tv_sec-ts.tv_sec)*1000.0 + (te.tv_nsec-ts.tv_nsec)/1e6;
        printf("CPU MatMul (%4dx%4d): %.3f ms\n", n, n, ms);
        free(A); free(B); free(C);
    }
    return 0;
}

In [ ]:
!gcc -O2 -o matmul_cpu task1_matmul_cpu.c && ./matmul_cpu

---
## Task 1b — CUDA Matrix Multiplication (Global Memory)
One thread per output element, accesses global memory directly. Block size: 16×16.

In [ ]:
%%writefile task1_matmul_global.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define BLOCK 16

__global__ void matmul_global(float *A, float *B, float *C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {
        float sum = 0.0f;
        for (int k=0; k<n; k++) sum += A[row*n+k] * B[k*n+col];
        C[row*n+col] = sum;
    }
}

void run(int n) {
    size_t bytes = n*n*sizeof(float);
    float *h_A=(float*)malloc(bytes), *h_B=(float*)malloc(bytes), *h_C=(float*)malloc(bytes);
    for (int i=0; i<n*n; i++) { h_A[i]=1.0f; h_B[i]=1.0f; }
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes); cudaMalloc(&d_B, bytes); cudaMalloc(&d_C, bytes);

    cudaEvent_t h2d_s, h2d_e, k_s, k_e, d2h_s, d2h_e;
    cudaEventCreate(&h2d_s); cudaEventCreate(&h2d_e);
    cudaEventCreate(&k_s);   cudaEventCreate(&k_e);
    cudaEventCreate(&d2h_s); cudaEventCreate(&d2h_e);

    cudaEventRecord(h2d_s);
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(h2d_e); cudaEventSynchronize(h2d_e);

    dim3 threads(BLOCK, BLOCK);
    dim3 blocks((n+BLOCK-1)/BLOCK, (n+BLOCK-1)/BLOCK);
    cudaEventRecord(k_s);
    matmul_global<<<blocks, threads>>>(d_A, d_B, d_C, n);
    cudaEventRecord(k_e); cudaEventSynchronize(k_e);

    cudaEventRecord(d2h_s);
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);
    cudaEventRecord(d2h_e); cudaEventSynchronize(d2h_e);

    float t_h2d, t_k, t_d2h;
    cudaEventElapsedTime(&t_h2d, h2d_s, h2d_e);
    cudaEventElapsedTime(&t_k,   k_s,   k_e);
    cudaEventElapsedTime(&t_d2h, d2h_s, d2h_e);
    printf("CUDA Global MatMul (%4dx%4d, block=%dx%d):\n", n, n, BLOCK, BLOCK);
    printf("  H->D: %.3f ms | Kernel: %.3f ms | D->H: %.3f ms | Total: %.3f ms\n\n",
           t_h2d, t_k, t_d2h, t_h2d+t_k+t_d2h);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
}

int main() {
    int sizes[] = {256, 512, 1024};
    for (int i=0; i<3; i++) run(sizes[i]);
    return 0;
}

In [ ]:
!nvcc -O2 -o matmul_global task1_matmul_global.cu && ./matmul_global

---
## Task 2 — CUDA Tiled Matrix Multiplication (Shared Memory)
Uses 16×16 shared-memory tiling to reduce global memory accesses. N = 1024.

In [ ]:
%%writefile task2_matmul_shared.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N    1024
#define TILE 16

__global__ void matmul_shared(float *A, float *B, float *C, int n) {
    __shared__ float tileA[TILE][TILE];
    __shared__ float tileB[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;
    for (int t=0; t<(n+TILE-1)/TILE; t++) {
        tileA[threadIdx.y][threadIdx.x] = (row<n && t*TILE+threadIdx.x<n) ? A[row*n+t*TILE+threadIdx.x] : 0.0f;
        tileB[threadIdx.y][threadIdx.x] = (col<n && t*TILE+threadIdx.y<n) ? B[(t*TILE+threadIdx.y)*n+col] : 0.0f;
        __syncthreads();
        for (int k=0; k<TILE; k++) sum += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        __syncthreads();
    }
    if (row<n && col<n) C[row*n+col] = sum;
}

int main() {
    int n = N;
    size_t bytes = n*n*sizeof(float);
    float *h_A=(float*)malloc(bytes), *h_B=(float*)malloc(bytes), *h_C=(float*)malloc(bytes);
    for (int i=0; i<n*n; i++) { h_A[i]=1.0f; h_B[i]=1.0f; }
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes); cudaMalloc(&d_B, bytes); cudaMalloc(&d_C, bytes);

    cudaEvent_t h2d_s, h2d_e, k_s, k_e, d2h_s, d2h_e;
    cudaEventCreate(&h2d_s); cudaEventCreate(&h2d_e);
    cudaEventCreate(&k_s);   cudaEventCreate(&k_e);
    cudaEventCreate(&d2h_s); cudaEventCreate(&d2h_e);

    cudaEventRecord(h2d_s);
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(h2d_e); cudaEventSynchronize(h2d_e);

    dim3 threads(TILE, TILE), blocks((n+TILE-1)/TILE, (n+TILE-1)/TILE);
    cudaEventRecord(k_s);
    matmul_shared<<<blocks, threads>>>(d_A, d_B, d_C, n);
    cudaEventRecord(k_e); cudaEventSynchronize(k_e);

    cudaEventRecord(d2h_s);
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);
    cudaEventRecord(d2h_e); cudaEventSynchronize(d2h_e);

    float t_h2d, t_k, t_d2h;
    cudaEventElapsedTime(&t_h2d, h2d_s, h2d_e);
    cudaEventElapsedTime(&t_k,   k_s,   k_e);
    cudaEventElapsedTime(&t_d2h, d2h_s, d2h_e);
    printf("CUDA Tiled MatMul (%dx%d, TILE=%d):\n", n, n, TILE);
    printf("  H->D: %.3f ms | Kernel: %.3f ms | D->H: %.3f ms | Total: %.3f ms\n",
           t_h2d, t_k, t_d2h, t_h2d+t_k+t_d2h);

    int ok=1;
    for (int i=0; i<n*n && ok; i++) if (h_C[i]!=(float)n) ok=0;
    printf("Verification: %s\n", ok?"PASSED":"FAILED");

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

In [ ]:
!nvcc -O2 -o matmul_shared task2_matmul_shared.cu && ./matmul_shared

---
## Task 4 — Parallel Reduction (Sum, Max, Min)
GPU shared-memory tree reduction vs. CPU sequential. Sizes: 100K, 1M, 10M, 100M.

In [ ]:
%%writefile task4_reduction.cu
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>
#include <time.h>
#include <cuda_runtime.h>

#define BLOCK 256

__global__ void reduce_sum(int *d_in, int *d_out, int n) {
    __shared__ int sdata[BLOCK];
    int tid=threadIdx.x, idx=blockIdx.x*blockDim.x*2+threadIdx.x;
    sdata[tid] = (idx<n?d_in[idx]:0) + (idx+BLOCK<n?d_in[idx+BLOCK]:0);
    __syncthreads();
    for (int s=BLOCK/2; s>0; s>>=1) { if(tid<s) sdata[tid]+=sdata[tid+s]; __syncthreads(); }
    if (tid==0) d_out[blockIdx.x]=sdata[0];
}

__global__ void reduce_max(int *d_in, int *d_out, int n) {
    __shared__ int sdata[BLOCK];
    int tid=threadIdx.x, idx=blockIdx.x*blockDim.x*2+threadIdx.x;
    sdata[tid]=INT_MIN;
    if(idx<n) sdata[tid]=max(sdata[tid],d_in[idx]);
    if(idx+BLOCK<n) sdata[tid]=max(sdata[tid],d_in[idx+BLOCK]);
    __syncthreads();
    for (int s=BLOCK/2; s>0; s>>=1) { if(tid<s) sdata[tid]=max(sdata[tid],sdata[tid+s]); __syncthreads(); }
    if (tid==0) d_out[blockIdx.x]=sdata[0];
}

__global__ void reduce_min(int *d_in, int *d_out, int n) {
    __shared__ int sdata[BLOCK];
    int tid=threadIdx.x, idx=blockIdx.x*blockDim.x*2+threadIdx.x;
    sdata[tid]=INT_MAX;
    if(idx<n) sdata[tid]=min(sdata[tid],d_in[idx]);
    if(idx+BLOCK<n) sdata[tid]=min(sdata[tid],d_in[idx+BLOCK]);
    __syncthreads();
    for (int s=BLOCK/2; s>0; s>>=1) { if(tid<s) sdata[tid]=min(sdata[tid],sdata[tid+s]); __syncthreads(); }
    if (tid==0) d_out[blockIdx.x]=sdata[0];
}

long long cpu_sum(int *a,int n){long long s=0;for(int i=0;i<n;i++)s+=a[i];return s;}
int cpu_max(int *a,int n){int m=INT_MIN;for(int i=0;i<n;i++)if(a[i]>m)m=a[i];return m;}
int cpu_min(int *a,int n){int m=INT_MAX;for(int i=0;i<n;i++)if(a[i]<m)m=a[i];return m;}

int gpu_reduce(int *d_in, int n, int op) {
    int blocks=(n+BLOCK*2-1)/(BLOCK*2);
    int *d_out; cudaMalloc(&d_out, blocks*sizeof(int));
    if(op==0) reduce_sum<<<blocks,BLOCK>>>(d_in,d_out,n);
    if(op==1) reduce_max<<<blocks,BLOCK>>>(d_in,d_out,n);
    if(op==2) reduce_min<<<blocks,BLOCK>>>(d_in,d_out,n);
    int *d_final; cudaMalloc(&d_final, sizeof(int));
    if(op==0) reduce_sum<<<1,BLOCK>>>(d_out,d_final,blocks);
    if(op==1) reduce_max<<<1,BLOCK>>>(d_out,d_final,blocks);
    if(op==2) reduce_min<<<1,BLOCK>>>(d_out,d_final,blocks);
    int result; cudaMemcpy(&result,d_final,sizeof(int),cudaMemcpyDeviceToHost);
    cudaFree(d_out); cudaFree(d_final);
    return result;
}

void run(int n) {
    printf("\n=== N = %d ===\n", n);
    int *h_in=(int*)malloc(n*sizeof(int));
    for(int i=0;i<n;i++) h_in[i]=rand()%1000;
    struct timespec ts,te;
    clock_gettime(CLOCK_MONOTONIC,&ts);
    long long csum=cpu_sum(h_in,n); int cmax=cpu_max(h_in,n); int cmin=cpu_min(h_in,n);
    clock_gettime(CLOCK_MONOTONIC,&te);
    double cpu_ms=(te.tv_sec-ts.tv_sec)*1000.0+(te.tv_nsec-ts.tv_nsec)/1e6;
    printf("CPU  sum=%lld  max=%d  min=%d  time=%.3f ms\n",csum,cmax,cmin,cpu_ms);
    int *d_in; cudaMalloc(&d_in,n*sizeof(int));
    cudaMemcpy(d_in,h_in,n*sizeof(int),cudaMemcpyHostToDevice);
    cudaEvent_t t0,t1; cudaEventCreate(&t0); cudaEventCreate(&t1);
    cudaEventRecord(t0);
    int gsum=gpu_reduce(d_in,n,0), gmax=gpu_reduce(d_in,n,1), gmin=gpu_reduce(d_in,n,2);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float gpu_ms; cudaEventElapsedTime(&gpu_ms,t0,t1);
    printf("GPU  sum=%d  max=%d  min=%d  time=%.3f ms\n",gsum,gmax,gmin,gpu_ms);
    printf("Speedup: %.2fx\n",cpu_ms/gpu_ms);
    printf("Verification sum: %s | max: %s | min: %s\n",
           (long long)gsum==csum?"OK":"FAIL",gmax==cmax?"OK":"FAIL",gmin==cmin?"OK":"FAIL");
    cudaFree(d_in); free(h_in);
}

int main() {
    int sizes[]={100000,1000000,10000000,100000000};
    for(int i=0;i<4;i++) run(sizes[i]);
    return 0;
}

In [ ]:
!nvcc -O2 -o reduction task4_reduction.cu && ./reduction

---
## Task 5 — Parallel Prefix Scan (Hillis-Steele)
Inclusive scan using shared-memory ping-pong. N = 1024 (single block).

In [ ]:
%%writefile task5_scan.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N     1024
#define BLOCK 1024

__global__ void hillis_steele_scan(int *d_in, int *d_out, int n) {
    extern __shared__ int temp[];
    int tid=threadIdx.x, pout=0, pin=1;
    temp[pout*n+tid] = (tid>0 ? d_in[tid-1] : 0);
    __syncthreads();
    for (int offset=1; offset<n; offset<<=1) {
        pout=1-pout; pin=1-pout;
        if(tid>=offset) temp[pout*n+tid]=temp[pin*n+tid]+temp[pin*n+tid-offset];
        else            temp[pout*n+tid]=temp[pin*n+tid];
        __syncthreads();
    }
    d_out[tid] = temp[pout*n+tid]+d_in[tid];
}

void cpu_scan(int *in, int *out, int n) {
    out[0]=in[0];
    for(int i=1;i<n;i++) out[i]=out[i-1]+in[i];
}

int main() {
    int n=N;
    int *h_in=(int*)malloc(n*sizeof(int)), *h_cpu=(int*)malloc(n*sizeof(int)), *h_gpu=(int*)malloc(n*sizeof(int));
    for(int i=0;i<n;i++) h_in[i]=rand()%10;
    struct timespec ts,te;
    clock_gettime(CLOCK_MONOTONIC,&ts); cpu_scan(h_in,h_cpu,n); clock_gettime(CLOCK_MONOTONIC,&te);
    double cpu_ms=(te.tv_sec-ts.tv_sec)*1000.0+(te.tv_nsec-ts.tv_nsec)/1e6;
    printf("CPU  scan (N=%d): %.4f ms\n",n,cpu_ms);

    int *d_in, *d_out;
    cudaMalloc(&d_in,n*sizeof(int)); cudaMalloc(&d_out,n*sizeof(int));
    cudaMemcpy(d_in,h_in,n*sizeof(int),cudaMemcpyHostToDevice);
    cudaEvent_t t0,t1; cudaEventCreate(&t0); cudaEventCreate(&t1);
    cudaEventRecord(t0);
    hillis_steele_scan<<<1,n,2*n*sizeof(int)>>>(d_in,d_out,n);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float gpu_ms; cudaEventElapsedTime(&gpu_ms,t0,t1);
    cudaMemcpy(h_gpu,d_out,n*sizeof(int),cudaMemcpyDeviceToHost);
    printf("GPU  scan (N=%d): %.4f ms\n",n,gpu_ms);
    printf("Speedup: %.2fx\n",cpu_ms/gpu_ms);
    int ok=1;
    for(int i=0;i<n;i++) if(h_cpu[i]!=h_gpu[i]){ok=0;break;}
    printf("Verification: %s\n",ok?"PASSED":"FAILED");
    printf("\nInput   : "); for(int i=0;i<10;i++) printf("%3d ",h_in[i]);
    printf("\nCPU out : "); for(int i=0;i<10;i++) printf("%3d ",h_cpu[i]);
    printf("\nGPU out : "); for(int i=0;i<10;i++) printf("%3d ",h_gpu[i]);
    printf("\n");
    cudaFree(d_in); cudaFree(d_out); free(h_in); free(h_cpu); free(h_gpu);
    return 0;
}

In [ ]:
!nvcc -O2 -o scan task5_scan.cu && ./scan

---
## Task 6 — CUDA Histogram
Compares CPU, GPU global-atomic, and GPU shared-memory histogram. 256 bins.

In [ ]:
%%writefile task6_histogram.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <cuda_runtime.h>

#define BINS  256
#define BLOCK 256

__global__ void hist_global(int *d_in, int *d_hist, int n) {
    int idx=blockIdx.x*blockDim.x+threadIdx.x;
    if(idx<n) atomicAdd(&d_hist[d_in[idx]],1);
}

__global__ void hist_shared(int *d_in, int *d_hist, int n) {
    __shared__ int s_hist[BINS];
    for(int i=threadIdx.x;i<BINS;i+=blockDim.x) s_hist[i]=0;
    __syncthreads();
    int idx=blockIdx.x*blockDim.x+threadIdx.x;
    if(idx<n) atomicAdd(&s_hist[d_in[idx]],1);
    __syncthreads();
    for(int i=threadIdx.x;i<BINS;i+=blockDim.x) atomicAdd(&d_hist[i],s_hist[i]);
}

void cpu_histogram(int *in, int *hist, int n) {
    memset(hist,0,BINS*sizeof(int));
    for(int i=0;i<n;i++) hist[in[i]]++;
}

void run(int n) {
    printf("\n=== N = %d, BINS = %d ===\n",n,BINS);
    int *h_in=(int*)malloc(n*sizeof(int));
    int *h_hist_cpu=(int*)malloc(BINS*sizeof(int)), *h_hist_gpu=(int*)malloc(BINS*sizeof(int));
    for(int i=0;i<n;i++) h_in[i]=rand()%BINS;
    struct timespec ts,te;
    clock_gettime(CLOCK_MONOTONIC,&ts); cpu_histogram(h_in,h_hist_cpu,n); clock_gettime(CLOCK_MONOTONIC,&te);
    double cpu_ms=(te.tv_sec-ts.tv_sec)*1000.0+(te.tv_nsec-ts.tv_nsec)/1e6;
    printf("CPU  histogram: %.3f ms\n",cpu_ms);
    int *d_in,*d_hist;
    cudaMalloc(&d_in,n*sizeof(int)); cudaMalloc(&d_hist,BINS*sizeof(int));
    cudaMemcpy(d_in,h_in,n*sizeof(int),cudaMemcpyHostToDevice);
    int grid=(n+BLOCK-1)/BLOCK;
    cudaEvent_t t0,t1; cudaEventCreate(&t0); cudaEventCreate(&t1);
    cudaMemset(d_hist,0,BINS*sizeof(int));
    cudaEventRecord(t0); hist_global<<<grid,BLOCK>>>(d_in,d_hist,n); cudaEventRecord(t1); cudaEventSynchronize(t1);
    float gm_ms; cudaEventElapsedTime(&gm_ms,t0,t1);
    cudaMemcpy(h_hist_gpu,d_hist,BINS*sizeof(int),cudaMemcpyDeviceToHost);
    int ok_gm=memcmp(h_hist_cpu,h_hist_gpu,BINS*sizeof(int))==0;
    printf("GPU  global atomic: %.3f ms  (Speedup: %.2fx)  Verify: %s\n",gm_ms,cpu_ms/gm_ms,ok_gm?"OK":"FAIL");
    cudaMemset(d_hist,0,BINS*sizeof(int));
    cudaEventRecord(t0); hist_shared<<<grid,BLOCK>>>(d_in,d_hist,n); cudaEventRecord(t1); cudaEventSynchronize(t1);
    float sm_ms; cudaEventElapsedTime(&sm_ms,t0,t1);
    cudaMemcpy(h_hist_gpu,d_hist,BINS*sizeof(int),cudaMemcpyDeviceToHost);
    int ok_sm=memcmp(h_hist_cpu,h_hist_gpu,BINS*sizeof(int))==0;
    printf("GPU  shared memory: %.3f ms  (Speedup: %.2fx)  Verify: %s\n",sm_ms,cpu_ms/sm_ms,ok_sm?"OK":"FAIL");
    cudaFree(d_in); cudaFree(d_hist); free(h_in); free(h_hist_cpu); free(h_hist_gpu);
}

int main() {
    int sizes[]={100000,1000000,10000000,100000000};
    for(int i=0;i<4;i++) run(sizes[i]);
    return 0;
}

In [ ]:
!nvcc -O2 -o histogram task6_histogram.cu && ./histogram

---
## Challenge Task — Optimized Reduction
Warp unrolling + two-element load per thread. Compares naive vs optimized on 10M elements.

In [ ]:
%%writefile challenge_reduction_optimized.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define BLOCK 256

__global__ void reduce_naive(int *d_in, int *d_out, int n) {
    __shared__ int sdata[BLOCK];
    int tid=threadIdx.x, idx=blockIdx.x*blockDim.x+threadIdx.x;
    sdata[tid]=(idx<n?d_in[idx]:0);
    __syncthreads();
    for(int s=BLOCK/2;s>0;s>>=1){if(tid<s)sdata[tid]+=sdata[tid+s];__syncthreads();}
    if(tid==0) d_out[blockIdx.x]=sdata[0];
}

__global__ void reduce_optimized(int *d_in, int *d_out, int n) {
    __shared__ int sdata[BLOCK];
    int tid=threadIdx.x, idx=blockIdx.x*blockDim.x*2+threadIdx.x;
    sdata[tid]=(idx<n?d_in[idx]:0)+(idx+BLOCK<n?d_in[idx+BLOCK]:0);
    __syncthreads();
    for(int s=BLOCK/2;s>32;s>>=1){if(tid<s)sdata[tid]+=sdata[tid+s];__syncthreads();}
    if(tid<32){
        volatile int *smem=sdata;
        smem[tid]+=smem[tid+32]; smem[tid]+=smem[tid+16];
        smem[tid]+=smem[tid+8];  smem[tid]+=smem[tid+4];
        smem[tid]+=smem[tid+2];  smem[tid]+=smem[tid+1];
    }
    if(tid==0) d_out[blockIdx.x]=sdata[0];
}

int run_reduce(int *d_in, int n, int optimized) {
    int blocks=optimized?(n+BLOCK*2-1)/(BLOCK*2):(n+BLOCK-1)/BLOCK;
    int *d_out; cudaMalloc(&d_out,blocks*sizeof(int));
    if(optimized) reduce_optimized<<<blocks,BLOCK>>>(d_in,d_out,n);
    else          reduce_naive<<<blocks,BLOCK>>>(d_in,d_out,n);
    int *d_final; cudaMalloc(&d_final,sizeof(int));
    if(optimized) reduce_optimized<<<1,BLOCK>>>(d_out,d_final,blocks);
    else          reduce_naive<<<1,BLOCK>>>(d_out,d_final,blocks);
    int result; cudaMemcpy(&result,d_final,sizeof(int),cudaMemcpyDeviceToHost);
    cudaFree(d_out); cudaFree(d_final);
    return result;
}

int main() {
    int n=10000000;
    int *h_in=(int*)malloc(n*sizeof(int));
    long long cpu_sum=0;
    for(int i=0;i<n;i++){h_in[i]=rand()%100;cpu_sum+=h_in[i];}
    int *d_in; cudaMalloc(&d_in,n*sizeof(int));
    cudaMemcpy(d_in,h_in,n*sizeof(int),cudaMemcpyHostToDevice);
    cudaEvent_t t0,t1; cudaEventCreate(&t0); cudaEventCreate(&t1);
    float ms;
    printf("CPU sum = %lld\n\n",cpu_sum);
    cudaEventRecord(t0); int r_naive=run_reduce(d_in,n,0); cudaEventRecord(t1); cudaEventSynchronize(t1);
    cudaEventElapsedTime(&ms,t0,t1);
    printf("Naive    reduction: %d  time=%.3f ms  match=%s\n",r_naive,ms,(long long)r_naive==cpu_sum?"YES":"NO");
    cudaEventRecord(t0); int r_opt=run_reduce(d_in,n,1); cudaEventRecord(t1); cudaEventSynchronize(t1);
    cudaEventElapsedTime(&ms,t0,t1);
    printf("Optimized reduction: %d  time=%.3f ms  match=%s\n",r_opt,ms,(long long)r_opt==cpu_sum?"YES":"NO");
    cudaFree(d_in); free(h_in);
    return 0;
}

In [ ]:
!nvcc -O2 -o reduction_opt challenge_reduction_optimized.cu && ./reduction_opt